In [1]:
import sys
sys.path.insert(0, "..")

import os
os.environ["PYTORCH_HIP_ALLOC_CONF"] = "garbage_collection_threshold:0.8"

from pathlib import Path
from datetime import datetime

from tqdm import tqdm
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import mlflow

from rustic_ml.encoding import encode_adsr, decode_adsr, NOTE_MIN, NOTE_MAX, N_NOTES, N_WAVEFORMS, WAVEFORMS
from rustic_ml.dataset import random_spec, render_mel, NpzDataset, prepare_dataloaders
from rustic_ml.analysis import MetricSpec, analyze_run
from rustic_ml.evaluation import (
    accumulate_note_waveform_inference,
    plot_note_accuracy, plot_waveform_accuracy,
    compare_audio_note_waveform,
)
from rustic_ml.mlflow_ui import show_register_widget, show_describe_widget, show_registered_models, setup_mlflow
from rustic_ml.training import set_seeds, log_hyperparams, setup_device
from rustic_ml.comparison import compare_note_waveform_models, load_registered_model

In [2]:
# MlFlow
MLFLOW_URI = "http://192.168.1.254:5000"
MLFLOW_EXPERIMENT = "WaveFormDualModel"

# Dataset — separate dir from notebook 1 (waveform labels, random waveforms)
DATA_DIR = Path("/data/datasets_waveform")

# Whether to train the note+waveform model or load a registered one
TRAIN_NOTE_WAVEFORM = True

# Training parameters
HYPER = {
    "seed":             42,
    "n_samples":        80_000,
    "batch_size_gen":   1_000,
    "batch_size":       64,
    "n_epochs":         80,
    "lr":               1e-3,
    "lambda_waveform":  1.0,   # weight of waveform CE loss relative to note CE loss
}
set_seeds(HYPER["seed"])

In [3]:
device = setup_device()
setup_mlflow(MLFLOW_URI, MLFLOW_EXPERIMENT)

CUDA detected — using NVIDIA GeForce GTX 1050
MLflow connected: http://192.168.1.254:5000  (experiment: WaveFormDualModel)


In [4]:
print(f"Using torch {torch.__version__} device {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"HIP version: {torch.version.hip}")
print(f"Using mlflow {mlflow.__version__}")
print(f"Rustic_ML OK - notes [{NOTE_MIN}, {NOTE_MAX}] N_NOTES={N_NOTES}")

Using torch 2.5.1+cu121 device NVIDIA GeForce GTX 1050
HIP version: None
Using mlflow 3.10.1
Rustic_ML OK - notes [36, 84] N_NOTES=49


## Dataset

Generates data with **random waveforms** (uniform over all 7 types) and saves waveform int labels alongside `mel`, `note`, and `adsr`.  
Uses a separate directory from notebook 1 to avoid mixing fixed-sine data with varied-waveform data.

In [5]:
train_loader, val_loader, train_ds, val_ds = prepare_dataloaders(
    data_dir=DATA_DIR,
    n_samples=HYPER["n_samples"],
    batch_size_gen=HYPER["batch_size_gen"],
    batch_size=HYPER["batch_size"],
    seed=HYPER["seed"],
    with_waveform=True,
)

Train: 64000 samples (64 files)  |  Val: 16000 samples (16 files)


## Model: NoteWaveformPredictor

Shared CNN backbone (4 conv blocks: 1→32→64→128→256 channels, BatchNorm + ReLU + MaxPool) followed by `AdaptiveAvgPool2d` and two independent classification heads:
- **Note head**: `Linear(256, N_NOTES)` — 49 classes (MIDI 36–84)
- **Waveform head**: `Linear(256, N_WAVEFORMS)` — 7 classes

In [6]:
from rustic_ml.models import NoteWaveformPredictor

model = NoteWaveformPredictor().to(device)

NoteWaveformPredictor - 403,192 trainable parameters


## Training

In [ ]:
if TRAIN_NOTE_WAVEFORM:
    optimizer = torch.optim.Adam(model.parameters(), lr=HYPER["lr"])
    ce_loss   = nn.CrossEntropyLoss()
    lam_wf    = HYPER["lambda_waveform"]

    with mlflow.start_run(run_name=f"NoteWaveformPredictor_{datetime.now():%Y%m%d_%H%M}") as run:
        RUN_ID = run.info.run_id
        log_hyperparams(HYPER)
        mlflow.log_param("model", "NoteWaveformPredictor")
        mlflow.log_param("n_notes", N_NOTES)
        mlflow.log_param("n_waveforms", N_WAVEFORMS)

        for epoch in tqdm(range(1, HYPER["n_epochs"] + 1)):
            # --- train ---
            model.train()
            t_loss_note, t_loss_wf, t_total, n_train = 0.0, 0.0, 0.0, 0
            for mel, note, _adsr, waveform in train_loader:
                mel      = mel.to(device)
                note     = torch.tensor(note, device=device) if not isinstance(note, torch.Tensor) else note.to(device)
                waveform = torch.tensor(waveform, device=device) if not isinstance(waveform, torch.Tensor) else waveform.to(device)
                note_idx = note - NOTE_MIN  # shift to 0-based class index

                note_logits, wf_logits = model(mel)
                loss_note = ce_loss(note_logits, note_idx)
                loss_wf   = ce_loss(wf_logits, waveform)
                loss      = loss_note + lam_wf * loss_wf

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                b = mel.size(0)
                t_loss_note += loss_note.item() * b
                t_loss_wf   += loss_wf.item()   * b
                t_total     += loss.item()       * b
                n_train     += b

            # --- val ---
            model.eval()
            v_loss_note, v_loss_wf, v_total, n_val = 0.0, 0.0, 0.0, 0
            v_note_correct, v_wf_correct = 0, 0
            with torch.no_grad():
                for mel, note, _adsr, waveform in val_loader:
                    mel      = mel.to(device)
                    note     = torch.tensor(note, device=device) if not isinstance(note, torch.Tensor) else note.to(device)
                    waveform = torch.tensor(waveform, device=device) if not isinstance(waveform, torch.Tensor) else waveform.to(device)
                    note_idx = note - NOTE_MIN

                    note_logits, wf_logits = model(mel)
                    loss_note = ce_loss(note_logits, note_idx)
                    loss_wf   = ce_loss(wf_logits, waveform)
                    loss      = loss_note + lam_wf * loss_wf

                    b = mel.size(0)
                    v_loss_note    += loss_note.item() * b
                    v_loss_wf      += loss_wf.item()   * b
                    v_total        += loss.item()       * b
                    n_val          += b
                    v_note_correct += (note_logits.argmax(1) == note_idx).sum().item()
                    v_wf_correct   += (wf_logits.argmax(1)   == waveform).sum().item()

            mlflow.log_metrics({
                "train/loss_note":      t_loss_note / n_train,
                "train/loss_waveform":  t_loss_wf   / n_train,
                "train/loss_total":     t_total     / n_train,
                "val/loss_note":        v_loss_note / n_val,
                "val/loss_waveform":    v_loss_wf   / n_val,
                "val/loss_total":       v_total     / n_val,
                "val/note_accuracy":    v_note_correct  / n_val,
                "val/waveform_accuracy": v_wf_correct   / n_val,
            }, step=epoch)

            if epoch % 10 == 0 or epoch == 1:
                print(
                    f"Epoch {epoch:3d}/{HYPER['n_epochs']}  "
                    f"loss_note={v_loss_note/n_val:.4f}  "
                    f"loss_wf={v_loss_wf/n_val:.4f}  "
                    f"note_acc={v_note_correct/n_val:.3f}  "
                    f"wf_acc={v_wf_correct/n_val:.3f}"
                )

        print(f"\nTraining complete. Run ID: {RUN_ID}")
        mlflow.pytorch.log_model(model, "note_waveform")

else:
    model = load_registered_model("NoteWaveformPredictor", device)
    RUN_ID = None
    print("Loaded registered NoteWaveformPredictor")

2026/03/24 13:35:25 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

  1%|▏         | 1/80 [01:43<2:16:43, 103.85s/it]

Epoch   1/80  loss_note=1.8982  loss_wf=0.2412  note_acc=0.553  wf_acc=0.853


 12%|█▎        | 10/80 [17:22<2:01:37, 104.25s/it]

Epoch  10/80  loss_note=1.8188  loss_wf=0.2854  note_acc=0.547  wf_acc=0.849


 25%|██▌       | 20/80 [34:45<1:44:15, 104.25s/it]

Epoch  20/80  loss_note=2.6985  loss_wf=0.2693  note_acc=0.577  wf_acc=0.852


 26%|██▋       | 21/80 [36:29<1:42:33, 104.30s/it]

## Loss curve analysis

In [ ]:
if RUN_ID is not None:
    analyze_run(
        metrics=[
            MetricSpec("loss_note",     "train", "loss"),
            MetricSpec("loss_note",     "val",   "loss"),
            MetricSpec("loss_waveform", "train", "loss"),
            MetricSpec("loss_waveform", "val",   "loss"),
            MetricSpec("loss_total",    "train", "loss"),
            MetricSpec("loss_total",    "val",   "loss"),
            MetricSpec("note_accuracy",     "val", "accuracy"),
            MetricSpec("waveform_accuracy", "val", "accuracy"),
        ],
        run_id=RUN_ID,
        tracking_uri=MLFLOW_URI,
    )

## Evaluation

In [ ]:
vals = accumulate_note_waveform_inference(model, val_loader, device)

plot_note_accuracy(vals)
plot_waveform_accuracy(vals)

In [ ]:
# Waveform confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(vals["waveform"]["targets"], vals["waveform"]["preds"])
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=WAVEFORMS).plot(ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title("Waveform confusion matrix (val set)")
plt.tight_layout()
if RUN_ID is not None:
    mlflow.start_run(run_id=RUN_ID, nested=True)
    mlflow.log_figure(fig, "waveform_confusion.png")
    mlflow.end_run()
plt.show()

## Audio comparison

Renders a few validation samples: shows mel spectrograms (ground truth vs predicted note+waveform) and playable audio widgets.  
ADSR is taken from ground truth since this model does not predict it.

In [ ]:
for idx in [0, 1, 2]:
    compare_audio_note_waveform(model, val_ds, device, sample_idx=idx, log_to_mlflow=(idx == 0))

## Comparison with previous registered version

In [ ]:
compare_note_waveform_models(
    model_name="NoteWaveformPredictor",
    current_model=model,
    loader=val_loader,
    device=device,
    tracking_uri=MLFLOW_URI,
)

## Register model

In [ ]:
if RUN_ID is not None:
    show_register_widget(RUN_ID, "model", default_name="NoteWaveformPredictor")
else:
    print("No run to register — set TRAIN_NOTE_WAVEFORM=True and re-run.")

In [ ]:
if RUN_ID is not None:
    show_describe_widget(RUN_ID)